In [ ]:
# Cell 1: install Hunyuan3D and dependencies. Run this first in Colab with GPU.
import os, sys
from pathlib import Path

os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["PYGLET_HEADLESS"] = "True"

REPO_DIR = Path("/content/Hunyuan3D-2")
if not REPO_DIR.exists():
    !git clone https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git /content/Hunyuan3D-2

%cd /content/Hunyuan3D-2

!apt-get -qq update
!apt-get -qq install -y libegl1-mesa libgles2-mesa mesa-utils
!pip -q install -r requirements.txt
!pip -q install -e .
!pip -q install --upgrade PyOpenGL PyOpenGL_accelerate pyrender==0.1.45 trimesh==4.4.1 open_clip_torch==2.24.0 fast_simplification open3d

# Hunyuan3D's requirements downgrade Pillow over Colab's preinstalled copy,
# which can leave a MIXED install (files from two versions) that breaks
# `import PIL` deep inside open_clip ("cannot import name '_Ink'").
# Force a clean single-version reinstall to fix that.
!pip -q install --force-reinstall pillow

sys.path.insert(0, str(REPO_DIR))
print("Setup complete.")
print("IMPORTANT: if this is the first run in this session, do Runtime -> Restart session,")
print("then re-run Cell 1 (it will be fast) and continue with Cell 2.")


In [ ]:
# Cell 2: multisignal evaluator code. Run this once after setup.
"""
Hunyuan3D mesh evaluator — multi-signal reliable scoring.

The single biggest reliability problem with image->mesh scoring is that the
input photo is ONE unknown viewpoint, while the rendered mesh is N viewpoints.
Comparing the photo against all N renders pollutes the score with viewpoint
mismatch noise that has nothing to do with mesh quality.

Design:
  1. Render 32 views (4 elevations x 8 azimuths, including below-horizon
     cameras so low-angle photos can also find a matching viewpoint).
  2. Find the K=3 renders whose silhouettes best match the input photo
     (selection signal: multi-scale IoU). These are the "matched views".
  3. Score CLIP/DINO/SigLIP semantic similarity ON THE MATCHED VIEWS only.
     To avoid selection bias ("best-of-N" optimism), the silhouette SCORE on
     those views uses an independent metric — boundary chamfer distance —
     not the IoU that picked them.
  4. Reduce the texture/color domain gap: renders are untextured, so ALL
     semantic models compare GRAYSCALE input vs grayscale renders (DINO
     proved NOT color-robust enough to skip this in practice).
  5. Use the FULL view set for consistency / blob / diversity checks.
  6. Geometry sanity check (faces, components, watertight, aspect, hull).
  7. Bootstrap stability: re-run the EXACT same estimator (top-K matched-view
     scoring) on random view subsets — measures fragility of the score that
     is actually reported.
  8. Cross-signal agreement + background-removal sanity -> reliability flag.

Outputs:
  - JSON with per-image full breakdown (rank percentile included)
  - Preview grid with per-view metrics overlaid
  - Spotlight image: input photo next to its top-3 matched renders
  - Saved mesh OBJ per image
  - Sortable HTML report

Calibration note: the map_*_to_100 ranges are heuristics. Run Cell 4
(meta-evaluation) once to verify the scorer actually ranks a good mesh above
degraded versions of itself before trusting absolute numbers.
"""

import gc
import html
import json
import logging
import os
import re
from pathlib import Path

import numpy as np
import open_clip
import pyrender
import torch
import trimesh
from PIL import Image, ImageDraw
from scipy.ndimage import binary_erosion, distance_transform_edt

from hy3dgen.rembg import BackgroundRemover
from hy3dgen.shapegen import (
    DegenerateFaceRemover,
    FaceReducer,
    FloaterRemover,
    Hunyuan3DDiTFlowMatchingPipeline,
)


# ============================================================================
# Constants
# ============================================================================

MODEL_ID = "tencent/Hunyuan3D-2mini"
SEED = 2025
RENDER_SIZE = 384
STEPS = 30

# 4 elevations x 8 azimuths = 32 views. Camera heights at radius 2.2; the
# negative one covers photos taken from below the object's midline.
ELEVATIONS = [-0.5, 0.0, 0.5, 1.0]
N_AZIMUTHS = 8
N_VIEWS = len(ELEVATIONS) * N_AZIMUTHS

# Matched-viewpoint scoring: how many top-IoU views to anchor on.
MATCHED_K = 3

# Multi-scale silhouette IoU search (catches scale mismatch between input
# and render).
IOU_SCALES = [0.85, 1.0, 1.15]

# Bootstrap stability check.
BOOTSTRAP_N = 8
BOOTSTRAP_SUBSET = 16

# Mask size used for silhouette comparison.
MASK_SIZE = 256

# Background-removal sanity range: foreground fraction outside this range
# means rembg probably failed and every downstream signal is untrustworthy.
FG_COVERAGE_RANGE = (0.02, 0.90)


# ============================================================================
# Calibration mappings — convert raw cosines / distances to 0..100 scale.
# Different models have very different baseline ranges; each gets its own.
# These are heuristics — validate the ordering with Cell 4 before trusting
# absolute values.
# ============================================================================

def score_from_range(x, low, high):
    return float(np.clip((x - low) / (high - low), 0.0, 1.0) * 100.0)


def map_clip_to_100(x):
    # Anchored to observed photo-vs-render cosines: a bad/wrong match sits
    # near 0.45, a strong match near 0.78.
    return score_from_range(x, 0.45, 0.80)


def map_dino_to_100(x):
    # DINOv2's global CLS cosine is COMPRESSED: ~0.42 is already a strong
    # photo-vs-untextured-render match and it rarely exceeds ~0.5. The old
    # (0.35, 0.80) range floored every good mesh at ~15 and caused the
    # "signals disagree" false alarm. Anchored to real data instead.
    return score_from_range(x, 0.15, 0.50)


def map_siglip_to_100(x):
    # SigLIP image-image cosines run high; anchored so a strong match (~0.83)
    # lands around 77 rather than saturating at 100.
    return score_from_range(x, 0.60, 0.90)


def map_iou_to_100(x):
    return float(np.clip(x, 0.0, 1.0) * 100.0)


def map_chamfer_to_100(d, size=MASK_SIZE):
    # Symmetric mean boundary distance in pixels on a size x size mask.
    # 0 px -> 100; >= 8% of the mask side -> 0.
    if not np.isfinite(d):
        return 0.0
    return float(np.clip(1.0 - d / (0.08 * size), 0.0, 1.0) * 100.0)


# ============================================================================
# Model loading
# ============================================================================

def setup_device():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)
    if device == "cuda":
        print("gpu:", torch.cuda.get_device_name(0))
    return device


def load_models(device, use_dino=True, use_siglip=True):
    rembg = BackgroundRemover()

    try:
        # the mini weights live in the hunyuan3d-dit-v2-mini subfolder
        shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            MODEL_ID, subfolder="hunyuan3d-dit-v2-mini")
    except Exception:
        shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(MODEL_ID)
    try:
        shape_pipe.to(device)
    except Exception:
        pass

    clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
        "ViT-H-14",
        pretrained="laion2b_s32b_b79k",
    )
    clip_model = clip_model.to(device).eval()
    print("clip: loaded ViT-H-14 (laion2b)")

    dino = None
    if use_dino:
        try:
            dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14").to(device).eval()
            print("dino: loaded dinov2_vitl14")
        except Exception as exc:
            print("dino: unavailable, continuing without it:", exc)

    siglip_model = None
    siglip_preprocess = None
    if use_siglip:
        try:
            siglip_model, _, siglip_preprocess = open_clip.create_model_and_transforms(
                "ViT-B-16-SigLIP-384",
                pretrained="webli",
            )
            siglip_model = siglip_model.to(device).eval()
            print("siglip: loaded ViT-B-16-SigLIP-384")
        except Exception as exc:
            print("siglip: unavailable, continuing without it:", exc)

    return rembg, shape_pipe, clip_model, clip_preprocess, dino, siglip_model, siglip_preprocess


# Module-level cache so evaluate_images (Cell 3) and meta_evaluate (Cell 4)
# share one set of loaded models instead of loading twice.
_MODEL_CACHE = {}


def get_models(device, use_dino=True, use_siglip=True):
    key = (device, use_dino, use_siglip)
    if key not in _MODEL_CACHE:
        _MODEL_CACHE[key] = load_models(device, use_dino=use_dino, use_siglip=use_siglip)
    return _MODEL_CACHE[key]


# ============================================================================
# Embeddings (batched — one forward pass per model for all views)
# ============================================================================

def to_grayscale_rgb(pil_img):
    """Collapse color so CLIP/SigLIP compare shape+shading, not palette.
    Renders are untextured; feeding a colored photo against a gray render
    lets color dominate the cosine."""
    return pil_img.convert("L").convert("RGB")


@torch.no_grad()
def clip_embed_batch(pil_imgs, model, preprocess, device, batch_size=32):
    feats = []
    for i in range(0, len(pil_imgs), batch_size):
        x = torch.stack([preprocess(im.convert("RGB")) for im in pil_imgs[i:i + batch_size]]).to(device)
        f = model.encode_image(x)
        feats.append(torch.nn.functional.normalize(f, dim=-1))
    return torch.cat(feats, 0)


@torch.no_grad()
def dino_embed_batch(pil_imgs, dino, device, size=224, batch_size=32):
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    feats = []
    for i in range(0, len(pil_imgs), batch_size):
        arrs = []
        for im in pil_imgs[i:i + batch_size]:
            im = to_grayscale_rgb(im)  # match the untextured render domain
            a = np.asarray(im.convert("RGB").resize((size, size)), dtype=np.float32) / 255.0
            arrs.append(((a - mean) / std).transpose(2, 0, 1))
        x = torch.from_numpy(np.stack(arrs)).to(device)
        f = dino(x)
        feats.append(torch.nn.functional.normalize(f, dim=-1))
    return torch.cat(feats, 0)


def clip_embed(pil_img, model, preprocess, device):
    return clip_embed_batch([pil_img], model, preprocess, device)[0]


def dino_embed(pil_img, dino, device):
    return dino_embed_batch([pil_img], dino, device)[0]


def cosine(a, b):
    return float((a * b).sum().clamp(-1, 1).item())


# ============================================================================
# Silhouette utilities
# ============================================================================

def to_mask(pil_img, threshold=0.5):
    arr = np.asarray(pil_img)
    if arr.ndim == 3 and arr.shape[-1] == 4:
        return arr[..., 3] > int(255 * threshold)
    if arr.ndim == 2:
        return arr < 245
    return np.any(arr[..., :3] < 245, axis=-1)


def foreground_coverage(pil_rgba):
    """Fraction of pixels rembg kept as foreground. Way outside a sane range
    means background removal failed — flag it instead of blaming the mesh."""
    alpha = np.asarray(pil_rgba.convert("RGBA"))[..., 3]
    return float((alpha > 16).mean())


def crop_normalize_mask(mask, size=MASK_SIZE):
    """Bounding-box crop + center on square + resize. Removes framing
    differences so two silhouettes from different cameras are comparable."""
    ys, xs = np.where(mask)
    if len(xs) == 0:
        return np.zeros((size, size), dtype=bool)
    y0, y1 = ys.min(), ys.max() + 1
    x0, x1 = xs.min(), xs.max() + 1
    cropped = mask[y0:y1, x0:x1]
    h, w = cropped.shape
    side = max(h, w)
    canvas = np.zeros((side, side), dtype=bool)
    yo = (side - h) // 2
    xo = (side - w) // 2
    canvas[yo:yo + h, xo:xo + w] = cropped
    pil = Image.fromarray(canvas.astype(np.uint8) * 255).resize((size, size), Image.NEAREST)
    return np.asarray(pil) > 127


def iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0


def multi_scale_iou(input_mask, render_mask, scales=IOU_SCALES):
    """Try a few relative scales and return the best IoU. Catches the case
    where the mesh has the right shape but slightly wrong proportions —
    fixed-scale IoU would unfairly punish it."""
    h, w = input_mask.shape
    best = 0.0
    for s in scales:
        if abs(s - 1.0) < 1e-6:
            scaled = render_mask
        else:
            new_h, new_w = max(1, int(h * s)), max(1, int(w * s))
            pil = Image.fromarray(render_mask.astype(np.uint8) * 255).resize((new_w, new_h), Image.NEAREST)
            arr = np.asarray(pil) > 127
            # Center on canvas of original size
            canvas = np.zeros_like(input_mask)
            ys = (h - new_h) // 2
            xs = (w - new_w) // 2
            ys_dst = max(0, ys)
            xs_dst = max(0, xs)
            ys_src = max(0, -ys)
            xs_src = max(0, -xs)
            cy = min(new_h - ys_src, h - ys_dst)
            cx = min(new_w - xs_src, w - xs_dst)
            if cy > 0 and cx > 0:
                canvas[ys_dst:ys_dst + cy, xs_dst:xs_dst + cx] = arr[ys_src:ys_src + cy, xs_src:xs_src + cx]
            scaled = canvas
        v = iou(input_mask, scaled)
        if v > best:
            best = v
    return best


def mask_boundary(mask):
    if not mask.any():
        return mask
    return mask & ~binary_erosion(mask)


def boundary_chamfer(a, b):
    """Symmetric mean boundary-to-boundary distance in pixels.

    Independent of the IoU that SELECTS matched views, so scoring with it
    avoids the best-of-N selection bias: IoU picks the viewpoint, chamfer
    judges how well the outline actually matches there."""
    ba, bb = mask_boundary(a), mask_boundary(b)
    if not ba.any() or not bb.any():
        return float("inf")
    da = distance_transform_edt(~ba)
    db = distance_transform_edt(~bb)
    return 0.5 * (float(da[bb].mean()) + float(db[ba].mean()))


# ============================================================================
# Rendering
# ============================================================================

def _look_at(camera_pos, target=np.zeros(3, dtype=np.float32), up=np.array([0, 1, 0], dtype=np.float32)):
    """Standard OpenGL look-at pose. pyrender cameras look along their LOCAL
    -Z axis, so the pose's +Z column must point from the target BACK toward
    the camera. (Pointing +Z at the target makes every camera face away from
    the object and renders nothing but background.)"""
    camera_pos = np.array(camera_pos, dtype=np.float32)
    z_axis = camera_pos - target
    z_axis = z_axis / (np.linalg.norm(z_axis) + 1e-8)
    x_axis = np.cross(up, z_axis)
    x_axis = x_axis / (np.linalg.norm(x_axis) + 1e-8)
    y_axis = np.cross(z_axis, x_axis)
    pose = np.eye(4, dtype=np.float32)
    pose[:3, :3] = np.stack([x_axis, y_axis, z_axis], axis=1)
    pose[:3, 3] = camera_pos
    return pose


def render_self_test(size=128):
    """Render a known-good icosphere and verify non-empty output — catches a
    broken EGL/OpenGL context. Uses _look_at (not an identity pose) so it
    also exercises the same camera-pose code path as the real renders."""
    try:
        m = trimesh.creation.icosphere(subdivisions=2)
        scene = pyrender.Scene(bg_color=[255, 255, 255, 0], ambient_light=[0.3] * 3)
        scene.add(pyrender.Mesh.from_trimesh(m, smooth=False))
        cam = pyrender.PerspectiveCamera(yfov=np.pi / 3.0)
        light = pyrender.DirectionalLight(color=np.ones(3), intensity=2.5)
        pose = _look_at([1.5, 0.8, 1.5])
        scene.add(cam, pose=pose)
        scene.add(light, pose=pose)
        r = pyrender.OffscreenRenderer(viewport_width=size, viewport_height=size)
        color, _ = r.render(scene, flags=pyrender.RenderFlags.RGBA)
        r.delete()
        alpha = color[..., 3]
        coverage = float((alpha > 10).mean())
        return coverage > 0.02, coverage
    except Exception as exc:
        print(f"render_self_test: exception {exc}")
        return False, 0.0


def render_views(mesh, size=RENDER_SIZE):
    mesh = mesh.copy()
    if not mesh.is_empty:
        mesh.apply_translation(-mesh.centroid)
        scale = np.max(mesh.extents) + 1e-8
        mesh.apply_scale(1.0 / scale)

    scene = pyrender.Scene(bg_color=[255, 255, 255, 0], ambient_light=[0.25] * 3)
    scene.add(pyrender.Mesh.from_trimesh(mesh, smooth=False))
    cam = pyrender.PerspectiveCamera(yfov=np.pi / 3.0)
    light = pyrender.DirectionalLight(color=np.ones(3), intensity=2.5)
    renderer = pyrender.OffscreenRenderer(viewport_width=size, viewport_height=size)

    rgb_views, rgba_views, view_meta = [], [], []
    radius = 2.2
    for ele_idx, elev in enumerate(ELEVATIONS):
        for az_idx in range(N_AZIMUTHS):
            theta = 2 * np.pi * (az_idx / N_AZIMUTHS)
            cam_pos = [radius * np.cos(theta), elev, radius * np.sin(theta)]
            pose = _look_at(cam_pos)
            nc = scene.add(cam, pose=pose)
            nl = scene.add(light, pose=pose)
            color, _ = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
            rgba = Image.fromarray(color.astype(np.uint8)).convert("RGBA")
            rgba_views.append(rgba)
            bg = Image.new("RGB", rgba.size, (255, 255, 255))
            bg.paste(rgba, (0, 0), rgba.split()[-1])
            rgb_views.append(bg)
            view_meta.append({"elev": float(elev), "azimuth_deg": float(np.degrees(theta))})
            scene.remove_node(nc)
            scene.remove_node(nl)
    renderer.delete()
    return rgb_views, rgba_views, view_meta


# ============================================================================
# Geometry sanity
# ============================================================================

def geometry_report(mesh):
    if mesh is None or mesh.is_empty or len(mesh.faces) == 0:
        return {
            "face_count": 0, "vertex_count": 0, "component_count": 0,
            "is_watertight": False, "aspect_ratio": None, "hull_ratio": None,
            "geometry_score_100": 0.0,
        }

    face_count = int(len(mesh.faces))
    vertex_count = int(len(mesh.vertices))
    try:
        component_count = len(mesh.split(only_watertight=False))
    except Exception:
        component_count = 99

    face_score = float(np.clip(face_count / 5000.0, 0.0, 1.0))
    component_score = float(np.clip(1.0 / max(1, component_count), 0.0, 1.0))
    watertight_score = 1.0 if mesh.is_watertight else 0.45

    try:
        extent = np.asarray(mesh.extents, dtype=np.float32)
        aspect = float(extent.max() / (extent.min() + 1e-8))
        aspect_score = float(np.clip(3.0 / max(3.0, aspect), 0.0, 1.0))
    except Exception:
        aspect = None
        aspect_score = 0.5

    # Hull ratio: penalize BOTH near-1 (sphere/cube blob) and near-0
    # (paper-thin / broken).
    hull_ratio = None
    hull_score = 0.65
    try:
        if mesh.is_watertight:
            hv = mesh.convex_hull.volume
            if hv > 0:
                hull_ratio = float(abs(mesh.volume) / hv)
                blob_pen = max(0.0, hull_ratio - 0.95) * 5.0
                thin_pen = max(0.0, 0.05 - hull_ratio) * 10.0
                hull_score = float(np.clip(1.0 - blob_pen - thin_pen, 0.0, 1.0))
    except Exception:
        pass

    score = (
        0.25 * face_score
        + 0.25 * component_score
        + 0.20 * watertight_score
        + 0.15 * aspect_score
        + 0.15 * hull_score
    ) * 100.0

    return {
        "face_count": face_count,
        "vertex_count": vertex_count,
        "component_count": int(component_count),
        "is_watertight": bool(mesh.is_watertight),
        "aspect_ratio": aspect,
        "hull_ratio": hull_ratio,
        "geometry_score_100": float(np.clip(score, 0.0, 100.0)),
    }


# ============================================================================
# Aggregation
# ============================================================================

def trimmed_mean(values, trim=0.1):
    """Drop top/bottom `trim` fraction. Robust to outlier views from
    e.g. self-occlusion or pathological angles."""
    s = np.sort(np.asarray(values, dtype=np.float32))
    if len(s) == 0:
        return 0.0
    drop = int(len(s) * trim)
    if drop > 0 and len(s) > 2 * drop:
        s = s[drop:-drop]
    return float(s.mean())


def matched_view_score(per_view_scores_100, ious_raw, k=MATCHED_K):
    """Mean of per-view scores on the K renders whose silhouettes best
    match the input. This is the heart of the design: it isolates the
    viewpoint that the photo was taken from, so the resulting CLIP/DINO
    score actually measures mesh quality, not viewpoint luck."""
    ious = np.asarray(ious_raw)
    k = min(k, len(ious))
    top_idx = np.argsort(ious)[-k:]
    return float(np.mean([per_view_scores_100[i] for i in top_idx])), top_idx.tolist()


def per_signal_consistency(scores_100):
    # Calibrated for shape-only renders: per-view variance is naturally high,
    # so we only penalize when views are REALLY bad, not merely below average.
    s = np.asarray(scores_100, dtype=np.float32)
    spread = float(np.max(s) - np.min(s))
    low_share = float(np.mean(s < 25.0))
    penalty = 0.0
    penalty += np.clip(spread / 65.0, 0.0, 1.0) * 6.0
    penalty += low_share * 10.0
    return float(np.clip(penalty, 0.0, 20.0))


def multi_signal_consistency(per_signal_scores):
    pens = [per_signal_consistency(s) for s in per_signal_scores if s is not None and len(s) > 0]
    if not pens:
        return 0.0
    return float(np.clip(np.mean(pens) * 1.1, 0.0, 25.0))


def inter_view_diversity(view_embeds):
    """view_embeds: [n, d] tensor of normalized embeddings."""
    if view_embeds is None or len(view_embeds) < 2:
        return 0.0
    sim = (view_embeds @ view_embeds.T).float().cpu().numpy()
    n = sim.shape[0]
    off_diag = sim[~np.eye(n, dtype=bool)]
    return float(1.0 - off_diag.mean())


def bootstrap_stability(per_view_signals, iou_raw, geometry_score, weights,
                        k=MATCHED_K, subset=BOOTSTRAP_SUBSET, n=BOOTSTRAP_N, seed=0):
    """Re-run the EXACT scoring estimator on random view subsets: within each
    subset, re-select the top-K matched views by IoU and average each signal
    on them — the same procedure as the reported score. High std = the
    reported number is fragile w.r.t. which viewpoints happened to be
    rendered."""
    rng = np.random.default_rng(seed)
    iou_raw = np.asarray(iou_raw)
    n_views = len(iou_raw)
    subset = min(subset, n_views)

    scores = []
    for _ in range(n):
        idx = rng.choice(n_views, size=subset, replace=False)
        kk = min(k, subset)
        top_local = idx[np.argsort(iou_raw[idx])[-kk:]]
        visual = 0.0
        for name, arr in per_view_signals.items():
            visual += weights[name] * float(np.asarray(arr)[top_local].mean())
        scores.append(0.72 * visual + 0.28 * geometry_score)

    arr = np.asarray(scores, dtype=np.float32)
    return float(arr.mean()), float(arr.std())


def reliability_flag(signal_scores, consistency, geometry_score, blob_penalty,
                     bootstrap_std, matched_iou, fg_coverage=None):
    """(flag, reasons[]). Thresholds loosened so realistic shape-only renders
    can actually reach 'high' reliability when the mesh is good."""
    available = [float(v) for v in signal_scores.values() if v is not None]
    spread = max(available) - min(available) if len(available) >= 2 else 0.0

    reasons = []
    fg_bad = (
        fg_coverage is not None
        and not (FG_COVERAGE_RANGE[0] <= fg_coverage <= FG_COVERAGE_RANGE[1])
    )
    if fg_bad:
        reasons.append(
            f"background removal looks wrong (foreground={fg_coverage:.0%}) — "
            "score reflects a bad input mask, not the mesh"
        )
    if spread > 45:
        reasons.append("signals disagree")
    if consistency > 18:
        reasons.append("some views score much worse")
    if geometry_score < 35:
        reasons.append("weak mesh geometry")
    if blob_penalty > 12:
        reasons.append("views look too similar")
    if bootstrap_std > 8:
        reasons.append("score unstable across view subsets")
    if matched_iou < 0.25:
        reasons.append("no rendered viewpoint matches the input silhouette well")

    if fg_bad:
        return "low", reasons
    if not reasons:
        return "high", reasons
    if (spread <= 50 and consistency <= 20 and geometry_score >= 30
            and bootstrap_std <= 10 and matched_iou >= 0.20):
        return "medium", reasons
    return "low", reasons


QUALITY_THRESHOLDS = (70.0, 50.0)  # >=70 resembles; 50-70 borderline; <50 poor


def quality_tier(final_score):
    """Score-based verdict on 'does the mesh resemble the input object'.

    THIS is the judgement to use. Unlike `reliability` (which only measures
    whether the signals AGREE, and was shown in the Cell 5 validation to not
    track quality at all), this tier is backed by the separation test:
    correct meshes land >=70, wrong/broken ones land well below 50.
    """
    hi, lo = QUALITY_THRESHOLDS
    if final_score >= hi:
        return "resembles"
    if final_score >= lo:
        return "borderline"
    return "poor"


def fragility_penalty(bootstrap_std, matched_iou):
    """Small score penalty for cases the reliability system already distrusts.

    Reliability remains the main warning. This only prevents a fragile score
    from ranking too highly when the matched viewpoint is weak or the metric
    swings a lot across view subsets.
    """
    bootstrap_penalty = float(np.clip((bootstrap_std - 6.0) / 8.0, 0.0, 1.0) * 8.0)
    viewpoint_penalty = float(np.clip((0.30 - matched_iou) / 0.20, 0.0, 1.0) * 10.0)
    return float(np.clip(bootstrap_penalty + viewpoint_penalty, 0.0, 18.0))


# ============================================================================
# Diagnostic outputs
# ============================================================================

def make_preview_grid(input_image, views, per_view_metrics, matched_idx, out_path):
    thumb_size = (192, 192)
    labeled = []

    first = input_image.copy().resize(thumb_size)
    ImageDraw.Draw(first).text((6, 6), "INPUT", fill=(255, 0, 0))
    labeled.append(first)

    matched_set = set(matched_idx)
    for idx, view in enumerate(views):
        im = view.copy().resize(thumb_size)
        # Highlight matched views with a red border
        if idx in matched_set:
            d = ImageDraw.Draw(im)
            for off in range(3):
                d.rectangle([off, off, thumb_size[0] - 1 - off, thumb_size[1] - 1 - off], outline=(255, 0, 0))
        d = ImageDraw.Draw(im)
        m = per_view_metrics[idx]
        d.text((6, 6), f"v{idx + 1}{' *' if idx in matched_set else ''}", fill=(255, 0, 0))
        d.text((6, 22), f"clip {m['clip']:.0f}", fill=(0, 0, 200))
        if m.get("dino") is not None:
            d.text((6, 38), f"dino {m['dino']:.0f}", fill=(0, 110, 0))
        if m.get("siglip") is not None:
            d.text((6, 54), f"sig  {m['siglip']:.0f}", fill=(180, 80, 0))
        d.text((6, 70), f"iou  {m['iou']:.0f}", fill=(160, 0, 160))
        d.text((6, 86), f"sil  {m['sil']:.0f}", fill=(120, 60, 0))
        labeled.append(im)

    cols = 6
    rows = int(np.ceil(len(labeled) / cols))
    grid = Image.new("RGB", (cols * thumb_size[0], rows * thumb_size[1]), "white")
    for idx, im in enumerate(labeled):
        x = (idx % cols) * thumb_size[0]
        y = (idx // cols) * thumb_size[1]
        grid.paste(im, (x, y))
    grid.save(out_path)


def make_spotlight(input_image, views, matched_idx, out_path):
    """Input photo next to its top-K matched renders. The single most useful
    diagnostic: a human can see at a glance whether the matched renders
    actually look like the input."""
    size = (320, 320)
    panels = [input_image.copy().resize(size)]
    ImageDraw.Draw(panels[0]).text((10, 10), "INPUT", fill=(255, 0, 0))
    for rank, idx in enumerate(matched_idx[::-1], 1):
        im = views[idx].copy().resize(size)
        ImageDraw.Draw(im).text((10, 10), f"matched #{rank} (v{idx + 1})", fill=(255, 0, 0))
        panels.append(im)
    out = Image.new("RGB", (size[0] * len(panels), size[1]), "white")
    for i, p in enumerate(panels):
        out.paste(p, (i * size[0], 0))
    out.save(out_path)


def write_html_report(results, out_path):
    rows = sorted(results, key=lambda r: r["final_score_100"], reverse=True)
    rows_html = []
    for rank, r in enumerate(rows, 1):
        reasons = "; ".join(r.get("review_reasons", [])) or "&mdash;"
        dino = "&mdash;" if r.get("dino_score_100") is None else f"{r['dino_score_100']:.1f}"
        siglip = "&mdash;" if r.get("siglip_score_100") is None else f"{r['siglip_score_100']:.1f}"
        rel = r["reliability"]
        color = {"high": "#2e7d32", "medium": "#f9a825", "low": "#c62828"}.get(rel, "#555")
        q = r.get("quality", quality_tier(r["final_score_100"]))
        qcolor = {"resembles": "#2e7d32", "borderline": "#f9a825", "poor": "#c62828"}.get(q, "#555")
        rows_html.append(
            f"<tr>"
            f"<td>{rank}</td>"
            f"<td>{html.escape(r['image'])}</td>"
            f"<td><b>{r['final_score_100']:.1f}</b></td>"
            f"<td style='color:{qcolor};font-weight:bold'>{q}</td>"
            f"<td style='color:{color};font-weight:bold'>{rel}</td>"
            f"<td>{r['clip_score_100']:.1f}</td>"
            f"<td>{dino}</td>"
            f"<td>{siglip}</td>"
            f"<td>{r['silhouette_score_100']:.1f}</td>"
            f"<td>{r['matched_iou_raw']:.2f}</td>"
            f"<td>{r['geometry_score_100']:.1f}</td>"
            f"<td>{r['bootstrap_std']:.2f}</td>"
            f"<td>{r.get('fragility_penalty', 0.0):.1f}</td>"
            f"<td>{html.escape(reasons)}</td>"
            f"<td><a href='{html.escape(Path(r['preview_path']).name)}'>preview</a> "
            f"<a href='{html.escape(Path(r['spotlight_path']).name)}'>spotlight</a></td>"
            f"</tr>"
        )
    html_doc = f"""<!doctype html>
<html><head><meta charset='utf-8'><title>Hunyuan3D evaluation</title>
<style>
body {{ font-family: system-ui, sans-serif; margin: 24px; }}
table {{ border-collapse: collapse; width: 100%; }}
th, td {{ border: 1px solid #ccc; padding: 6px 10px; text-align: left; }}
th {{ background: #f0f0f0; cursor: pointer; }}
tr:nth-child(even) {{ background: #fafafa; }}
small {{ color: #666; }}
</style></head><body>
<h1>Hunyuan3D mesh evaluation</h1>
<p><small>Sorted by final_score (descending). Click preview/spotlight links to inspect.
<br>sil = boundary-chamfer silhouette score on the matched views (independent of the
IoU that selected them, so it is not a best-of-N optimistic estimate).
match.iou = raw IoU of the matched views — low means no rendered viewpoint
resembles the photo's viewpoint.
<br>Reliability = high means CLIP, DINO, SigLIP and the silhouette signals all agree.
Low reliability = open the spotlight image and judge by eye.
<br><b>quality</b> (resembles / borderline / poor) is the score-based verdict to
trust: >=70 resembles the input, 50-70 borderline, &lt;50 poor. The
<b>reliability</b> column is debug-only (signal agreement) and does NOT track
quality.</small></p>
<table>
<thead><tr>
<th>#</th><th>image</th><th>final</th><th>quality</th><th>reliability</th>
<th>clip</th><th>dino</th><th>siglip</th><th>sil</th><th>match.iou</th>
<th>geom</th><th>boot.std</th><th>frag.pen</th><th>reasons</th><th>view</th>
</tr></thead>
<tbody>
{''.join(rows_html)}
</tbody></table>
</body></html>"""
    Path(out_path).write_text(html_doc, encoding="utf-8")


# ============================================================================
# Pipeline
# ============================================================================

@torch.no_grad()
def make_mesh_from_image(img_path, rembg, shape_pipe, device, steps=STEPS):
    img = Image.open(img_path).convert("RGB").resize((1024, 1024))
    img_fg = rembg(img)
    g = torch.Generator(device=device).manual_seed(SEED)
    out = shape_pipe(image=img_fg, num_inference_steps=steps, mc_algo="mc", generator=g)[0]
    out = FloaterRemover()(out)
    out = DegenerateFaceRemover()(out)
    out = FaceReducer()(out)
    if isinstance(out, trimesh.Trimesh):
        return out, img_fg
    if hasattr(out, "as_trimesh"):
        return out.as_trimesh(), img_fg
    return out, img_fg


def score_one(img_path, rembg_img, mesh,
              clip_model, clip_preprocess,
              dino,
              siglip_model, siglip_preprocess,
              device, out_dir):

    inp_rgba = rembg_img.convert("RGBA") if isinstance(rembg_img, Image.Image) else \
        Image.open(img_path).convert("RGBA")
    inp_white = Image.new("RGB", inp_rgba.size, (255, 255, 255))
    inp_white.paste(inp_rgba, (0, 0), inp_rgba.split()[-1])
    inp_mask = crop_normalize_mask(to_mask(inp_rgba))
    fg_coverage = foreground_coverage(inp_rgba)

    rgb_views, rgba_views, view_meta = render_views(mesh)
    n = len(rgb_views)

    # Grayscale for ALL semantic models (renders are untextured; color must
    # not drive the cosine). DINO's inputs are grayscaled inside
    # dino_embed_batch, so it takes the original views here.
    inp_gray = to_grayscale_rgb(inp_white)
    gray_views = [to_grayscale_rgb(v) for v in rgb_views]

    # ---- Batched embeddings: one forward pass per model for all views ----
    inp_clip = clip_embed(inp_gray, clip_model, clip_preprocess, device)
    view_clip = clip_embed_batch(gray_views, clip_model, clip_preprocess, device)
    clip_raw = (view_clip @ inp_clip).clamp(-1, 1).cpu().numpy()
    clip_scores = [map_clip_to_100(float(c)) for c in clip_raw]

    dino_raw = None
    dino_scores = None
    view_dino = None
    if dino is not None:
        inp_dino = dino_embed(inp_white, dino, device)
        view_dino = dino_embed_batch(rgb_views, dino, device)
        dino_raw = (view_dino @ inp_dino).clamp(-1, 1).cpu().numpy()
        dino_scores = [map_dino_to_100(float(d)) for d in dino_raw]

    siglip_raw = None
    siglip_scores = None
    if siglip_model is not None:
        inp_siglip = clip_embed(inp_gray, siglip_model, siglip_preprocess, device)
        view_siglip = clip_embed_batch(gray_views, siglip_model, siglip_preprocess, device)
        siglip_raw = (view_siglip @ inp_siglip).clamp(-1, 1).cpu().numpy()
        siglip_scores = [map_siglip_to_100(float(s)) for s in siglip_raw]

    diversity_embeds = view_dino if view_dino is not None else view_clip

    # ---- Silhouettes: IoU (selection signal) + boundary chamfer (scoring) ----
    iou_raw, iou_scores, chamfer_scores = [], [], []
    for rgba in rgba_views:
        render_mask = crop_normalize_mask(to_mask(rgba))
        iou_val = multi_scale_iou(inp_mask, render_mask)
        iou_raw.append(iou_val)
        iou_scores.append(map_iou_to_100(iou_val))
        chamfer_scores.append(map_chamfer_to_100(boundary_chamfer(inp_mask, render_mask)))

    clip_arr = np.asarray(clip_scores, dtype=np.float32)
    iou_arr = np.asarray(iou_scores, dtype=np.float32)
    iou_raw_arr = np.asarray(iou_raw, dtype=np.float32)
    chamfer_arr = np.asarray(chamfer_scores, dtype=np.float32)
    dino_arr = np.asarray(dino_scores, dtype=np.float32) if dino_scores else None
    siglip_arr = np.asarray(siglip_scores, dtype=np.float32) if siglip_scores else None

    # ---- Diagnostic: catch blank renders before they silently produce 0 ----
    if float(iou_raw_arr.max()) < 0.01:
        raise RuntimeError(
            f"All {n} rendered views are empty for {Path(img_path).name}. "
            "This usually means the offscreen renderer (pyrender/EGL) is broken "
            "in this Colab session, or the generated mesh has no faces. "
            "Try: Runtime -> Restart runtime, then re-run Cell 1, 2, 3 in order. "
            f"(mesh face_count={len(mesh.faces)}, is_empty={mesh.is_empty})"
        )

    # ---- KEY: matched-viewpoint scoring ----
    # IoU SELECTS the K=3 best-matching viewpoints; CLIP/DINO/SigLIP and the
    # chamfer silhouette score JUDGE quality on them. Selection and scoring
    # use different signals to avoid best-of-N selection bias.
    matched_clip_score, matched_idx = matched_view_score(clip_arr, iou_raw_arr)
    matched_dino_score = None
    if dino_arr is not None:
        matched_dino_score, _ = matched_view_score(dino_arr, iou_raw_arr)
    matched_siglip_score = None
    if siglip_arr is not None:
        matched_siglip_score, _ = matched_view_score(siglip_arr, iou_raw_arr)

    # Silhouette score: boundary chamfer on the matched views (NOT the IoU
    # that picked them — that would be a best-of-N optimistic estimate).
    silhouette_score_100 = float(np.mean([chamfer_arr[i] for i in matched_idx]))
    matched_iou_raw = float(np.mean([iou_raw_arr[i] for i in matched_idx]))

    # ---- Final visual score: matched-view fusion ----
    # DINO carries the most weight among semantic signals: untextured renders
    # make CLIP/SigLIP noisier even after the grayscale trick.
    if dino_arr is None and siglip_arr is None:
        weights = {"clip": 0.35, "silhouette": 0.65}
        per_view_for_boot = {"clip": clip_arr, "silhouette": chamfer_arr}
        visual_score_100 = 0.35 * matched_clip_score + 0.65 * silhouette_score_100
    elif dino_arr is not None and siglip_arr is None:
        weights = {"clip": 0.15, "dino": 0.30, "silhouette": 0.55}
        per_view_for_boot = {"clip": clip_arr, "dino": dino_arr, "silhouette": chamfer_arr}
        visual_score_100 = (0.15 * matched_clip_score + 0.30 * matched_dino_score
                            + 0.55 * silhouette_score_100)
    elif dino_arr is None and siglip_arr is not None:
        weights = {"clip": 0.25, "siglip": 0.15, "silhouette": 0.60}
        per_view_for_boot = {"clip": clip_arr, "siglip": siglip_arr, "silhouette": chamfer_arr}
        visual_score_100 = (0.25 * matched_clip_score + 0.15 * matched_siglip_score
                            + 0.60 * silhouette_score_100)
    else:
        # Stage-1 meshes are untextured -> geometry-led scoring. The boundary
        # chamfer silhouette dominates; DINO stays as the strongest "is it the
        # same object" check (covers the silhouette's blind spot on interior
        # structure); CLIP/SigLIP are weak assists.
        weights = {"clip": 0.10, "dino": 0.25, "siglip": 0.10, "silhouette": 0.55}
        per_view_for_boot = {"clip": clip_arr, "dino": dino_arr,
                             "siglip": siglip_arr, "silhouette": chamfer_arr}
        visual_score_100 = (
            0.10 * matched_clip_score
            + 0.25 * matched_dino_score
            + 0.10 * matched_siglip_score
            + 0.55 * silhouette_score_100
        )

    geo = geometry_report(mesh)

    consistency = multi_signal_consistency([clip_arr, chamfer_arr, dino_arr, siglip_arr])
    diversity = inter_view_diversity(diversity_embeds)
    blob_penalty = float(np.clip((0.05 - diversity) * 500.0, 0.0, 20.0)) if diversity < 0.05 else 0.0

    boot_mean, boot_std = bootstrap_stability(
        per_view_for_boot, iou_raw_arr, geo["geometry_score_100"], weights
    )

    fragility = fragility_penalty(boot_std, matched_iou_raw)
    final = (
        0.72 * visual_score_100
        + 0.28 * geo["geometry_score_100"]
        - consistency
        - blob_penalty
        - fragility
    )
    final = float(np.clip(final, 0.0, 100.0))

    signal_scores = {
        "clip": matched_clip_score,
        "dino": matched_dino_score,
        "siglip": matched_siglip_score,
        "silhouette": silhouette_score_100,
    }
    flag, reasons = reliability_flag(
        signal_scores, consistency, geo["geometry_score_100"],
        blob_penalty, boot_std, matched_iou_raw, fg_coverage,
    )

    # ---- Diagnostic outputs ----
    out_dir.mkdir(parents=True, exist_ok=True)
    stem = Path(img_path).stem
    per_view = []
    for i in range(n):
        per_view.append({
            "clip": float(clip_arr[i]),
            "dino": None if dino_arr is None else float(dino_arr[i]),
            "siglip": None if siglip_arr is None else float(siglip_arr[i]),
            "iou": float(iou_arr[i]),
            "sil": float(chamfer_arr[i]),
        })

    preview_path = out_dir / f"{stem}_preview.jpg"
    spotlight_path = out_dir / f"{stem}_spotlight.jpg"
    mesh_path = out_dir / f"{stem}.obj"

    make_preview_grid(inp_white, rgb_views, per_view, matched_idx, preview_path)
    make_spotlight(inp_white, rgb_views, matched_idx, spotlight_path)
    try:
        mesh.export(str(mesh_path))
    except Exception as exc:
        print(f"  warning: could not save mesh OBJ: {exc}")
        mesh_path = None

    return {
        "image": Path(img_path).name,
        "final_score_100": final,
        "quality": quality_tier(final),
        "reliability": flag,
        "review_reasons": reasons,
        "visual_score_100": visual_score_100,
        # Matched-view (primary) scores:
        "clip_score_100": matched_clip_score,
        "dino_score_100": matched_dino_score,
        "siglip_score_100": matched_siglip_score,
        "silhouette_score_100": silhouette_score_100,
        "matched_view_indices": matched_idx,
        "matched_iou_raw": matched_iou_raw,
        "input_fg_coverage": fg_coverage,
        # Geometry + reliability diagnostics:
        "geometry_score_100": geo["geometry_score_100"],
        "consistency_penalty": consistency,
        "blob_penalty": blob_penalty,
        "fragility_penalty": fragility,
        "inter_view_diversity": diversity,
        "bootstrap_mean": boot_mean,
        "bootstrap_std": boot_std,
        # Full-view raw / per-view data:
        "clip_per_view_100": clip_arr.tolist(),
        "silhouette_per_view_100": chamfer_arr.tolist(),
        "iou_per_view_100": iou_arr.tolist(),
        "dino_per_view_100": None if dino_arr is None else dino_arr.tolist(),
        "siglip_per_view_100": None if siglip_arr is None else siglip_arr.tolist(),
        "iou_per_view_raw": iou_raw_arr.tolist(),
        "clip_per_view_cosine": [float(c) for c in clip_raw],
        "dino_per_view_cosine": None if dino_raw is None else [float(d) for d in dino_raw],
        "siglip_per_view_cosine": None if siglip_raw is None else [float(s) for s in siglip_raw],
        "view_meta": view_meta,
        # File outputs:
        "preview_path": str(preview_path),
        "spotlight_path": str(spotlight_path),
        "mesh_path": str(mesh_path) if mesh_path else None,
        **{k: v for k, v in geo.items() if k != "geometry_score_100"},
    }


def evaluate_images(image_paths, output_dir="hunyuan_eval_outputs",
                    use_dino=True, use_siglip=True, resume=True):
    # The library registers loggers under BOTH names (upstream code uses the
    # typo'd "shapgen" in places) — silence both.
    logging.getLogger("hy3dgen.shapgen").setLevel(logging.ERROR)
    logging.getLogger("hy3dgen.shapegen").setLevel(logging.ERROR)
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    os.environ.setdefault("PYGLET_HEADLESS", "True")

    out_dir = Path(output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    json_path = out_dir / "hunyuan_multisignal_scores.json"

    # Resume support
    existing = {}
    if resume and json_path.exists():
        try:
            for r in json.loads(json_path.read_text(encoding="utf-8")):
                existing[r["image"]] = r
            if existing:
                print(f"resume: loaded {len(existing)} existing scores from {json_path}")
        except Exception:
            pass

    device = setup_device()
    rembg, shape_pipe, clip_model, clip_preprocess, dino, siglip_model, siglip_preprocess = \
        get_models(device, use_dino=use_dino, use_siglip=use_siglip)

    # Renderer sanity check — fail loudly here instead of silently producing
    # zero scores for every image.
    ok, coverage = render_self_test()
    print(f"renderer self-test: {'OK' if ok else 'FAILED'} (alpha coverage = {coverage:.3f})")
    if not ok:
        raise RuntimeError(
            "Offscreen renderer is producing blank frames. This is almost always "
            "a Colab EGL/OpenGL context problem. FIX: Runtime -> Restart runtime, "
            "then re-run Cell 1 (apt install must complete BEFORE pyrender is "
            "imported), then Cell 2, then Cell 3 — in that exact order."
        )

    results = list(existing.values())
    seen = set(existing.keys())

    for idx, img_path in enumerate(image_paths, 1):
        name = Path(img_path).name
        if name in seen:
            print(f"\n[{idx}/{len(image_paths)}] {name}  (cached)")
            continue
        print(f"\n[{idx}/{len(image_paths)}] {name}")
        try:
            mesh, rembg_img = make_mesh_from_image(img_path, rembg, shape_pipe, device)
            result = score_one(img_path, rembg_img, mesh,
                               clip_model, clip_preprocess,
                               dino,
                               siglip_model, siglip_preprocess,
                               device, out_dir)
        except Exception as exc:
            print(f"  ERROR: {exc}")
            continue

        results.append(result)
        seen.add(name)

        dino_str = "n/a" if result["dino_score_100"] is None else f"{result['dino_score_100']:.0f}"
        siglip_str = "n/a" if result["siglip_score_100"] is None else f"{result['siglip_score_100']:.0f}"
        print(
            f"  final={result['final_score_100']:.2f}  "
            f"reliability={result['reliability']}  "
            f"(matched_clip={result['clip_score_100']:.0f} "
            f"dino={dino_str} siglip={siglip_str} "
            f"sil={result['silhouette_score_100']:.0f}) "
            f"boot_std={result['bootstrap_std']:.1f}"
        )
        if result["review_reasons"]:
            print("  review:", ", ".join(result["review_reasons"]))
        print(f"  spotlight: {result['spotlight_path']}")

        # Stream-write so a crash mid-batch doesn't lose results.
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

        del mesh
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Add rank percentiles (calibration-free ranking signal).
    if results:
        scores = [r["final_score_100"] for r in results]
        order = np.argsort(np.argsort(scores))  # 0 = lowest, n-1 = highest
        n_r = len(scores)
        for i, r in enumerate(results):
            r["rank_percentile"] = float((order[i] / max(1, n_r - 1)) * 100.0)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    html_path = out_dir / "report.html"
    write_html_report(results, html_path)
    print("\nSaved:", json_path)
    print("Report:", html_path)
    print("\nINTERPRETATION:")
    print("  reliability=high  -> trust the score; signals agree, score is stable")
    print("  reliability=medium -> ranking probably right; absolute number may be noisy")
    print("  reliability=low   -> open the SPOTLIGHT image; signals disagree or mesh is off")
    return results


# ============================================================================
# Meta-evaluation (used by Cell 4): does the scorer rank good above bad?
# ============================================================================

def _degraded_variants(mesh, seed=0):
    """Build known-worse versions of a mesh. If the scorer can't rank the
    original above these, its numbers should not be trusted."""
    rng = np.random.default_rng(seed)
    scale = float(np.max(mesh.extents))
    variants = {}

    for name, sigma in [("noisy_2pct", 0.02), ("noisy_5pct", 0.05)]:
        m = mesh.copy()
        m.vertices = m.vertices + rng.normal(0.0, sigma * scale, m.vertices.shape)
        variants[name] = m

    try:
        variants["blob_convex_hull"] = mesh.convex_hull
    except Exception as exc:
        print(f"meta: skipping convex hull variant: {exc}")

    sphere = trimesh.creation.icosphere(subdivisions=3)
    sphere.apply_scale(scale / (np.max(sphere.extents) + 1e-8))
    variants["wrong_sphere"] = sphere

    target = max(200, int(len(mesh.faces) * 0.02))
    try:
        try:
            variants["decimated_2pct"] = mesh.simplify_quadric_decimation(face_count=target)
        except TypeError:
            variants["decimated_2pct"] = mesh.simplify_quadric_decimation(target)
    except Exception as exc:
        print(f"meta: skipping decimation variant (install fast_simplification?): {exc}")

    return variants


def meta_evaluate(img_path, output_dir="/content/hunyuan_meta_eval",
                  use_dino=True, use_siglip=True):
    """Generate ONE mesh, degrade it in known ways, score everything with the
    exact production scorer, and check the ordering. PASS on all checks is a
    prerequisite for trusting the calibration constants."""
    device = setup_device()
    rembg, shape_pipe, clip_model, clip_preprocess, dino, siglip_model, siglip_preprocess = \
        get_models(device, use_dino=use_dino, use_siglip=use_siglip)

    ok, coverage = render_self_test()
    print(f"renderer self-test: {'OK' if ok else 'FAILED'} (alpha coverage = {coverage:.3f})")
    if not ok:
        raise RuntimeError("Offscreen renderer broken — restart runtime, re-run Cells 1-2.")

    print("generating mesh from", Path(img_path).name)
    mesh, rembg_img = make_mesh_from_image(img_path, rembg, shape_pipe, device)

    variants = {"original": mesh}
    variants.update(_degraded_variants(mesh))

    out_root = Path(output_dir)
    scores = {}
    for name, m in variants.items():
        sub_dir = out_root / re.sub(r"[^A-Za-z0-9_-]", "_", name)
        try:
            r = score_one(img_path, rembg_img, m,
                          clip_model, clip_preprocess,
                          dino,
                          siglip_model, siglip_preprocess,
                          device, sub_dir)
            scores[name] = r["final_score_100"]
            print(f"  {name:20s} final={r['final_score_100']:6.2f}  reliability={r['reliability']}")
        except Exception as exc:
            print(f"  {name:20s} ERROR: {exc}")

    print("\nMETA-EVALUATION CHECKS (original should beat every degraded variant):")
    checks = []
    for name in scores:
        if name != "original":
            checks.append((f"original > {name}", scores.get("original", -1) > scores[name]))
    if "noisy_2pct" in scores and "noisy_5pct" in scores:
        checks.append(("noisy_2pct > noisy_5pct (monotonic in damage)",
                       scores["noisy_2pct"] > scores["noisy_5pct"]))

    n_pass = 0
    for label, passed in checks:
        print(f"  [{'PASS' if passed else 'FAIL'}] {label}")
        n_pass += int(passed)

    print(f"\n{n_pass}/{len(checks)} checks passed.")
    if n_pass == len(checks):
        print("Scorer ordering looks sane — calibration constants are usable for RANKING.")
    else:
        print("Some checks FAILED — do not trust absolute scores; inspect the failing")
        print("variants' spotlight images and consider adjusting map_*_to_100 ranges.")
    return scores

print("Multisignal evaluator loaded. Now run Cell 3 (evaluation) or Cell 4 (meta-evaluation).")


In [ ]:
# Cell 3: ask how many images, upload them, evaluate, then print scores one by one.
from google.colab import files
from pathlib import Path
from IPython.display import display, Image as DisplayImage

EXPECTED_IMAGES = int(input("How many images do you want to upload/evaluate? ").strip())
if EXPECTED_IMAGES <= 0:
    raise ValueError("Please enter a positive number.")

print(f"Upload exactly {EXPECTED_IMAGES} image(s). Supported: png, jpg, jpeg, webp")
uploaded = files.upload()

valid_exts = {".png", ".jpg", ".jpeg", ".webp"}
img_paths = []
for name in uploaded.keys():
    p = Path("/content/Hunyuan3D-2") / name
    if p.suffix.lower() in valid_exts:
        img_paths.append(str(p))

img_paths = sorted(img_paths)
if len(img_paths) != EXPECTED_IMAGES:
    raise RuntimeError(
        f"Expected {EXPECTED_IMAGES} image(s), but found {len(img_paths)} valid image file(s). "
        "Run this cell again and upload the right files."
    )

print("Images ready:")
for p in img_paths:
    print(" -", Path(p).name)

OUTPUT_DIR = "/content/hunyuan_eval_outputs"
results = evaluate_images(
    img_paths,
    output_dir=OUTPUT_DIR,
    use_dino=True,
    use_siglip=True,
    resume=False,
)

# Use chr(10) for literal newlines so an editor/linter cannot accidentally
# rewrite an escaped string into a multi-line literal (which would crash).
NL = chr(10)
BAR = "=" * 20
print(NL + BAR + " FINAL SCORES " + BAR)
sorted_results = sorted(results, key=lambda x: x["final_score_100"], reverse=True)
for r in sorted_results:
    dino = "n/a" if r.get("dino_score_100") is None else f"{r['dino_score_100']:.1f}"
    siglip = "n/a" if r.get("siglip_score_100") is None else f"{r['siglip_score_100']:.1f}"
    reasons = "; ".join(r.get("review_reasons", [])) or "none"
    line1 = f"{r['image']}: {r['final_score_100']:.2f}/100 | verdict={r.get('quality','?')}  (reliability={r['reliability']}, debug)"
    line2 = f"  clip={r['clip_score_100']:.1f}, dino={dino}, siglip={siglip}, sil={r['silhouette_score_100']:.1f}, geom={r['geometry_score_100']:.1f}"
    line3 = f"  review: {reasons}"
    print(line1)
    print(line2)
    print(line3)

print(NL + "Saved outputs:")
print(f" - JSON: {OUTPUT_DIR}/hunyuan_multisignal_scores.json")
print(f" - HTML report: {OUTPUT_DIR}/report.html")
print(f" - preview/spotlight images and OBJ meshes: {OUTPUT_DIR}")

print(NL + "Spotlight previews (input photo next to its top-3 matched renders):")
for r in sorted_results:
    header = f"{r['image']} | final={r['final_score_100']:.2f} | reliability={r['reliability']}"
    print(NL + header)
    if r.get("spotlight_path"):
        display(DisplayImage(filename=r["spotlight_path"]))


In [ ]:
# Cell 4 (optional, run once): meta-evaluation — verify the scorer can tell
# good meshes from bad ones BEFORE trusting its numbers.
#
# It generates ONE mesh from an image you upload, then scores deliberately
# degraded versions of that same mesh (vertex noise, heavy decimation, convex
# hull "blob", a plain sphere) with the exact production scorer. If the
# original doesn't outrank every degraded variant, the calibration constants
# (map_*_to_100 ranges, fusion weights) need adjusting — and absolute scores
# from Cell 3 should be treated as ranking-only.
from google.colab import files
from pathlib import Path

print("Upload ONE image for the meta-evaluation (png/jpg/jpeg/webp).")
uploaded = files.upload()

valid_exts = {".png", ".jpg", ".jpeg", ".webp"}
meta_img = None
for name in uploaded.keys():
    p = Path("/content/Hunyuan3D-2") / name
    if p.suffix.lower() in valid_exts:
        meta_img = str(p)
        break
if meta_img is None:
    raise RuntimeError("No valid image uploaded. Run this cell again.")

meta_scores = meta_evaluate(meta_img, output_dir="/content/hunyuan_meta_eval")


In [ ]:
# Cell 5 (validation): is low/medium/high ACTUALLY trustworthy?
#
# Calibrating on good meshes alone can make everything read "high" without
# proving anything - the same way a bug made everything read "low". The only
# honest test is SEPARATION: do correct meshes measurably beat wrong ones?
# This cell builds negative controls and reports whether they do. It can, and
# will, print FAIL if the metric doesn't discriminate.
#
#   positives : each mesh scored against ITS OWN photo   (should score high)
#   cross-neg : each mesh scored against OTHER photos     (should score lower)
#   degraded  : noise / decimation / hull-blob / sphere   (should score lower)
#
# Tip: upload images of DIFFERENT objects. Cross-scoring near-identical objects
# (e.g. five similar baskets) is a deliberately hard test - if the metric
# can't separate them, that is a TRUE finding about its limits, not a failure
# of this cell.

from google.colab import files
from pathlib import Path
from collections import Counter
import numpy as np
import logging
logging.getLogger("hy3dgen.shapgen").setLevel(logging.ERROR)
logging.getLogger("hy3dgen.shapegen").setLevel(logging.ERROR)

print("Upload 2+ images (different objects give the strongest cross-control).")
uploaded = files.upload()
valid = {".png", ".jpg", ".jpeg", ".webp"}
paths = sorted(str(Path("/content/Hunyuan3D-2") / n) for n in uploaded
               if Path(n).suffix.lower() in valid)
assert len(paths) >= 2, "Need at least 2 images for the cross negative-control."

device = setup_device()
rembg, shape_pipe, clip_m, clip_pp, dino, sig_m, sig_pp = get_models(device)

ok, cov = render_self_test()
assert ok, "Renderer broken - restart runtime and re-run Cells 1-2."

# Generate each mesh once (the expensive step); reuse for every scoring below.
meshes = {}
for p in paths:
    print("generating mesh:", Path(p).name)
    meshes[p] = make_mesh_from_image(p, rembg, shape_pipe, device)

VROOT = Path("/content/hunyuan_validation")

def score(mesh, photo_path, rembg_img, tag):
    return score_one(photo_path, rembg_img, mesh,
                     clip_m, clip_pp, dino, sig_m, sig_pp, device, VROOT / tag)

positives, cross = [], []
for i, pi in enumerate(paths):
    mesh_i, rembg_i = meshes[pi]
    positives.append((Path(pi).name, score(mesh_i, pi, rembg_i, f"pos_{i}")))
    for j, pj in enumerate(paths):
        if i == j:
            continue
        mesh_j, _ = meshes[pj]                       # wrong mesh vs photo i
        cross.append((f"{Path(pj).name}->{Path(pi).name}",
                      score(mesh_j, pi, rembg_i, f"cross_{i}_{j}")))

# Degradation controls on the first mesh.
mesh0, rembg0 = meshes[paths[0]]
degraded = [(name, score(m, paths[0], rembg0, f"deg_{name}"))
            for name, m in _degraded_variants(mesh0).items()]

def stats(rows):
    fs = [r["final_score_100"] for _, r in rows]
    return min(fs), sum(fs) / len(fs), max(fs)

print("\n================= SEPARATION =================")
for label, rows in [("POSITIVE (self)", positives),
                    ("CROSS (wrong mesh)", cross),
                    ("DEGRADED", degraded)]:
    lo, mn, hi = stats(rows)
    print(f"{label:20} n={len(rows):2}  final: min={lo:5.1f}  mean={mn:5.1f}  max={hi:5.1f}")

pos_min = min(r["final_score_100"] for _, r in positives)
neg_max = max(r["final_score_100"] for _, r in (cross + degraded))
margin = pos_min - neg_max
print(f"\nSeparation margin (worst positive - best negative) = {margin:+.1f}")

print("quality on positives:", dict(Counter(r["quality"] for _, r in positives)))
print("quality on negatives:", dict(Counter(r["quality"] for _, r in cross + degraded)))
print("(reliability flag is debug-only) pos:",
      dict(Counter(r["reliability"] for _, r in positives)),
      "neg:", dict(Counter(r["reliability"] for _, r in cross + degraded)))

print("\n================= VERDICT =================")
if margin > 5:
    print("PASS: correct meshes clearly beat wrong/broken ones. The score")
    print("      discriminates, so the ranking and the flags carry real signal.")
elif margin > 0:
    print("WEAK: positives edge out negatives but the margin is thin. Use scores")
    print("      as ranking-only; do NOT trust small gaps or the high/med/low flag.")
else:
    print("FAIL: a wrong or broken mesh scored >= a correct one. As calibrated the")
    print("      metric is NOT reliable and the flags are meaningless. Re-fit the")
    print("      map_*_to_100 LOW bounds to the negative cosines below, HIGH bounds")
    print("      to the positive cosines, update Cell 2, and re-run this cell.")

# Data-driven bound suggestions: LOW from wrong-mesh cosines, HIGH from correct.
def matched_cosines(rows, key):
    out = []
    for _, r in rows:
        v = r.get(key)
        if v:
            out += [v[i] for i in r["matched_view_indices"]]
    return out

print("\nSuggested calibration ranges (from THIS run's positives vs negatives):")
for key, name in [("clip_per_view_cosine", "CLIP"),
                  ("dino_per_view_cosine", "DINO"),
                  ("siglip_per_view_cosine", "SigLIP")]:
    pos_c, neg_c = matched_cosines(positives, key), matched_cosines(cross, key)
    if pos_c and neg_c:
        lo, hi = np.percentile(neg_c, 75), np.percentile(pos_c, 90)
        print(f"  {name:6}: wrong-mesh~{np.mean(neg_c):.2f}  correct~{np.mean(pos_c):.2f}"
              f"   -> map_{name.lower()}_to_100 range ~({lo:.2f}, {hi:.2f})")
